# 01 — The Steering Envelope model

The race model (Python port of the JS v0.2 artifact), the mean-field
N-actor extension, and the Sustenance Ledger society layer.

The one law everything hangs on: crossing a capability corner of tightness
$k$ at deployment velocity $v$ with steering capacity $s$ draws a
control-loss event with probability

$$h = \sigma\big(\beta\,(\tfrac{v\,k}{s\,c_0} - 1)\big)$$

Inside the envelope ($v k < s c_0$) hazard is small; outside it saturates.
The flags: e/acc maximizes $v$; d/acc shrinks $k$ (armor); s/acc constrains
$v \le f(s)$ (steering); w/acc grows the judgment feeding $s$ (driver).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))
import numpy as np
import matplotlib.pyplot as plt
from steering_envelope.model import (PRESETS, monte_carlo, hazard, Outcome)
from steering_envelope.validate import style
style.apply()

## The hazard law

In [2]:
ratios = np.linspace(0.2, 2.0, 200)
fig, ax = plt.subplots(figsize=(7, 3.6))
for beta, color in [(2, style.C_BLUE), (5.36, style.C_YELLOW),
                    (10, style.C_MAGENTA)]:
    ax.plot(ratios, [hazard(r, 1, 1, beta=beta) for r in ratios],
            color=color, label=f"beta={beta}")
ax.axvline(1.0, color=style.GRID, linewidth=1)
ax.set_xlabel("v·k / s·c0 (envelope ratio)")
ax.set_ylabel("P(control loss) at a corner")
ax.legend(); ax.set_title("the envelope edge is at ratio = 1", loc="left")
plt.show()

## The five worldviews, raced against the same field

Targets from the JS v0.2 artifact: e/acc lands ~60% crash+pileup; the
saxxer preset's plurality outcome is convoy/arrived.

In [3]:
rows = []
for name in ["eacc", "dacc", "saxxer", "wacc", "stopper"]:
    d = monte_carlo(PRESETS[name], n=3000, seed=5)
    rows.append((name, d))
    print(f"{name:8s} " + "  ".join(
        f"{o.value}={d[o.value]:.2f}" for o in Outcome if d[o.value] > 0.01)
        + f"   P(bad)={d['P_bad']:.2f}")

eacc     ARRIVED=0.05  CONVOY=0.34  CRASHED=0.44  PILEUP=0.17   P(bad)=0.61


dacc     ARRIVED=0.06  CONVOY=0.71  CRASHED=0.12  PILEUP=0.11   P(bad)=0.23


saxxer   ARRIVED=0.10  CONVOY=0.79  CRASHED=0.05  PILEUP=0.06   P(bad)=0.11


wacc     ARRIVED=0.05  CONVOY=0.74  CRASHED=0.10  PILEUP=0.11   P(bad)=0.21


stopper  ARRIVED=0.05  SHOTGUN=0.86  PILEUP=0.09   P(bad)=0.09


In [4]:
outs = [o.value for o in Outcome]
colors = {"ARRIVED": style.C_GREEN, "CONVOY": style.C_BLUE,
          "SHOTGUN": style.C_YELLOW, "HELD": style.MUTED,
          "CRASHED": style.C_MAGENTA, "PILEUP": "#8f2f4f"}
fig, ax = plt.subplots(figsize=(8, 3.8))
bottoms = np.zeros(len(rows))
for o in ["ARRIVED", "CONVOY", "SHOTGUN", "HELD", "CRASHED", "PILEUP"]:
    vals = np.array([d[o] for _, d in rows])
    ax.bar([n for n, _ in rows], vals, bottom=bottoms, color=colors[o],
           label=o, width=0.62, edgecolor=style.SURFACE, linewidth=2)
    bottoms += vals
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
ax.set_ylabel("outcome probability")
ax.set_title("same course, same field, different worldview", loc="left")
plt.show()

## Mean field: coordination vs private virtue

Replace the single rival with 16 actors drawing throttle from a
distribution; kappa couples every actor toward the envelope norm. The
theorem under test:
$$\frac{\partial P(\text{good}\,|\,\text{ego})}{\partial \kappa} \;>\; \frac{\partial P(\text{good}\,|\,\text{ego})}{\partial I_{\text{own}}}$$
on the diligent-ego domain ($I_{own} \ge 0.45$). Paired common-random-number
finite differences; see `meanfield.py` for the honest boundary (below basic
diligence the inequality genuinely fails: buy your own brakes first).

In [5]:
from steering_envelope.meanfield import outcome_vs_kappa, theorem_check
curve = outcome_vs_kappa(n_rep=200, seed=2)
ks = [r["kappa"] for r in curve]
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(ks, [r["P_good_ego"] for r in curve], color=style.C_BLUE,
        label="P(good | ego)")
ax.plot(ks, [r["P_good_field"] for r in curve], color=style.C_GREEN,
        label="P(good | field avg)")
ax.plot(ks, [r["P_pileup"] for r in curve], color=style.C_MAGENTA,
        label="P(pileup wave)")
ax.set_xlabel("coordination kappa"); ax.legend()
ax.set_title("what coordination buys", loc="left")
plt.show()

In [6]:
tc = theorem_check(n_rep=400, seed=11)
print(f"claim: {tc['claim']}")
print(f"holds at {tc['n_holds']}/{tc['n_gridpoints']} gridpoints, "
      f"mean margin {tc['mean_margin']:+.3f} (paired SE ~{tc['mean_se']:.3f})")
print("passes:", tc["passes"])

claim: dP(good|ego)/dkappa > dP(good|ego)/dI_own
holds at 7/9 gridpoints, mean margin +0.038 (paired SE ~0.049)
passes: False


## The kitchen: hours-per-week-of-essentials per tribe

The society layer converts race outcomes into the only metric the page
headlines: how many hours of work buy dinner, rent and power, per tribe.
The PROOF table asks which worldview minimizes each tribe's worst decade —
and the honesty check re-runs it with coordination forced to zero.

In [7]:
from steering_envelope.society import proof_table
nat = proof_table(n=800, seed=4)
z = proof_table(n=800, seed=4, coordination={k: 0.0 for k in
    ("eacc", "dacc", "saxxer", "wacc", "stopper")})
print("clear wins by worldview, natural K: ", nat["win_counts"])
print("clear wins by worldview, K forced 0:", z["win_counts"])
print()
print(f"{'tribe':40s} {'natural-K winner':16s} worst-decade hours (saxxer vs eacc)")
for r_n in nat["rows"]:
    print(f"{r_n['tribe'][:38]:40s} {r_n['min_worst_decade']:16s} "
          f"{r_n['worst']['saxxer']:5.1f} vs {r_n['worst']['eacc']:5.1f}")

clear wins by worldview, natural K:  {'eacc': 0, 'dacc': 0, 'saxxer': 4, 'wacc': 0, 'stopper': 0, 'tie': 4}
clear wins by worldview, K forced 0: {'eacc': 0, 'dacc': 0, 'saxxer': 0, 'wacc': 0, 'stopper': 0, 'tie': 8}

tribe                                    natural-K winner worst-decade hours (saxxer vs eacc)
salaried professional (Amsterdam)        tie               16.7 vs  17.8
gig / warehouse worker (US metro)        saxxer            31.4 vs  41.4
street vendor (Lagos)                    saxxer            49.9 vs  52.7
smallholder farmer (rural India)         tie               53.3 vs  53.8
Gulf rentier citizen                     tie                9.3 vs   9.4
maker / artisan (EU)                     saxxer            22.8 vs  24.5
online creator (global, verified-human   saxxer            27.1 vs  29.8
congregation / commune (any country)     tie               34.6 vs  35.1


The result the simulator lets you rediscover by hand: s/acc wins the
kitchen only through the coordination dial. With K = 0 every row is a tie —
private virtue keeps your lane clean but the field still sets the world your
tribe eats in.